# מפגש 14 — פתרון מורה מבוקר
מממש את מלוא מטלת המקור: island/seed17, shapes עם triangle חדשה, ומסלול numeric+categorical. אין מודל חיזוי ואין GridSearch.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
url='https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv'
penguins_df=pd.read_csv(url)
original=penguins_df.copy(deep=True)
model_df=penguins_df.dropna(subset=['species']).copy()
X=model_df[['island']]
y=model_df['species']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=17,stratify=y)
assert penguins_df.shape==(344,7)
assert model_df.shape==(344,7)
assert X_train.shape==(258,1) and X_test.shape==(86,1)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

## קידוד island על Train בלבד. שמות העמודות מסודרים לפי האימון.

In [ ]:
encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False).set_output(transform='pandas')
encoder.fit(X_train)
X_train_encoded=encoder.transform(X_train)
X_test_encoded=encoder.transform(X_test)
print('Training categories',encoder.categories_)
print('Columns',X_train_encoded.columns.tolist())
display(X_train_encoded.head(3),X_test_encoded.head(3))
assert encoder.categories_[0].tolist()==['Biscoe','Dream','Torgersen']
assert X_train_encoded.columns.tolist()==['island_Biscoe','island_Dream','island_Torgersen']
assert X_train_encoded.shape==(258,3) and X_test_encoded.shape==(86,3)
assert X_train_encoded.index.equals(y_train.index)
assert X_test_encoded.index.equals(y_test.index)
assert np.all(X_train_encoded.to_numpy().sum(axis=1)==1)
assert penguins_df.equals(original)

## ערך חדש toy_triangle אינו מקודד לקטגוריה מוכרת. אין Fit על Test.

In [ ]:
toy_train=pd.DataFrame({'shape':['circle','square','circle']})
toy_test=pd.DataFrame({'shape':['triangle','square']})
toy_encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False).set_output(transform='pandas')
toy_encoder.fit(toy_train)
toy_test_encoded=toy_encoder.transform(toy_test)
display(toy_test_encoded)
assert toy_test_encoded.columns.tolist()==['shape_circle','shape_square']
assert np.array_equal(toy_test_encoded.to_numpy(),[[0,0],[0,1]])

## רשות — מיפוי סדרי
מיפוי 0,1,2 מבטא סדר בלבד; מרווחים שווים הם הנחה נוספת.

In [ ]:
levels_df=pd.DataFrame({'level':['נמוך','בינוני','גבוה']})
level_map={'נמוך':0,'בינוני':1,'גבוה':2}
levels_df['order']=levels_df['level'].map(level_map)
display(levels_df)
assert levels_df['order'].tolist()==[0,1,2]

## הכנה משולבת — AGE מספרי ו־GROUP שמי
חציון [5,15] הוא 10, אחרי השלמה [5,15,10], ממוצע 10 וסטיית תקן √(50/3). שלוש עמודות פלט, New(E) זרה.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
lab_train=pd.DataFrame({'age':[5.,15.,None],'group':['C','D','D']})
lab_new=pd.DataFrame({'age':[10.],'group':['E']})
raw_lab=lab_train.copy(deep=True)
prepare=ColumnTransformer([
 ('numeric',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),['age']),
 ('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['group'])
])
lab_train_ready=prepare.fit_transform(lab_train)
lab_new_ready=prepare.transform(lab_new)
print('Features:',prepare.get_feature_names_out().tolist())
print('Train:',lab_train_ready)
print('New:',lab_new_ready)
assert prepare.get_feature_names_out().tolist()==['numeric__age','categorical__group_C','categorical__group_D']
assert lab_train_ready.shape==(3,3) and lab_new_ready.shape==(1,3)
assert np.allclose(lab_train_ready[:,0],[-np.sqrt(1.5),np.sqrt(1.5),0])
assert np.array_equal(lab_train_ready[:,1:],[[1,0],[0,1],[0,1]])
assert np.allclose(lab_new_ready,[[0,0,0]])
assert lab_train.equals(raw_lab)

## משמעות ותשובת הגנה
ב־One-Hot אפסים משמעו לא הקטגוריה המסומנת; בגיל מתוקנן אפס משמעו השוואה לממוצע האימון. שילוב שתי הכנות לא מאמן מסווג. המדיניות handle_unknown='ignore' אינה שומרת הבחנה בין קטגוריות לא מוכרות שונות.